# Dataset
All kept chunks of `outputs/chunks/<podcast>/merged_speakers_30s/` (made in `chunk_clean`) in one folder. The `rejected/` chunks are not used.
Output: `dataset/chunks/` + `dataset/metadata.jsonl`. Each run rebuilds the dataset from scratch.

In [1]:
import json
import shutil
from collections import Counter
from pathlib import Path

import soundfile as sf
import torch
import torchaudio.functional as AF
from tqdm.auto import tqdm

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
chunks_root = root / "outputs" / "chunks"
SRC_FOLDER = "merged_speakers_30s"  # the final chunks of chunk_clean
dataset_dir = root / "dataset"
wav_dir = dataset_dir / "chunks"
OUT_SR = 48000  # every chunk is resampled to this (the chunks are 16 kHz)
# sharp filter: torchaudio's default one takes ~3-4 dB off the top of the band (7-8 kHz: s, sh), this one keeps it
RESAMPLE = dict(lowpass_filter_width=64, rolloff=0.99, resampling_method="sinc_interp_kaiser", beta=14.769656459379492)
ABSOLUTE_PATHS = False  # False: audio_filepath = "chunks/<name>.wav" (relative to dataset/), True: full path

# every kept chunk = every row of a manifest.jsonl (chunk_clean rewrites it with the kept chunks only)
podcasts, items = [], []  # items: (podcast number, podcast title, source wav, manifest row)
for src_dir in sorted(chunks_root.glob(f"*/{SRC_FOLDER}")):
    if not (src_dir / "manifest.jsonl").exists():
        print("skip (no manifest):", src_dir.parent.name)
        continue
    podcasts.append(src_dir.parent.name)
    rows = [json.loads(l) for l in (src_dir / "manifest.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
    for r in rows:
        # the manifest path is absolute -> rebuild it from the folder, so a moved project still works
        items.append((len(podcasts), src_dir.parent.name, src_dir / Path(r["audio_filepath"]).name, r))


def new_name(num, src):
    """File names repeat across podcasts (every one has 0000_speaker_0.wav) -> podcast number in front."""
    return f"{num:03d}_{src.name}"


missing = [src for _, _, src, _ in items if not src.exists()]
names = Counter(new_name(num, src) for num, _, src, _ in items)
assert not missing, f"{len(missing)} chunks in a manifest have no wav, e.g. {missing[:3]}"
assert max(names.values()) == 1, "two chunks would get the same name"
print(f"{len(items)} chunks from {len(podcasts)} podcasts, "
      f"{sum(r['duration'] for *_, r in items) / 3600:.2f} h by the manifests")
for num, title in enumerate(podcasts, 1):
    print(f"  {num:03d}  {title}")

/Users/macbook/Documents/IT/projects/tts/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


14983 chunks from 31 podcasts, 36.77 h by the manifests
  001  18-35 yosh bo'lsangiz, shu 8ta darsni eshitishingiz kerak
  002  2026-yilda Google'ga ishga kirish XARITASI
  003  28-son - GTA VI, Fransiyadagi noroziliklar, Tesla uchar avtomobili va boshqa yangiliklar tahlili
  004  29-son - Krinjdan tortib siyosatgacha hafta davomida sodir bo‘lgan voqealar muhokamasi
  005  31 - son - Birinçi besoqol, musulmon, tub amerikalik senatorlar. Amerika siyosati kök rangda
  006  40 daqiqada yillar davomida yo'qotganimizni qaytaramiz - Maqolim 1-son
  007  50 yoshida ham studentlikni tanlaydigan odam bilan suhbat
  008  99% odamlar buni eplay olmaydi - Iqtibos podcast #27
  009  AQLI PASTlar avlodi (bo'lib qolyapmiz)
  010  Afsus qilmaslik uchun bugun nima qilish kerak - Hasanxon Yahyo Abdulmajid
  011  Chegaralar torayayotganini his qildim - Jamshidxon Ziyoxonov
  012  Dasturchilik faoliyatimni yakunlayman! - Azim Pulat Gashtak podkastida
  013  Ertaga bankrot bo‘lmaslik uchun hozir nima qilis

In [2]:
wav_dir.mkdir(parents=True, exist_ok=True)
for old in wav_dir.glob("*.wav"):  # leftovers from an earlier run
    old.unlink()

STATS = ("speech_sec", "noise_dbfs", "snr_db", "other_sound_pct")  # measured in chunk_clean
with open(dataset_dir / "metadata.jsonl", "w", encoding="utf-8") as f:
    for num, title, src, r in tqdm(items, unit="file"):
        dst = wav_dir / new_name(num, src)
        x, sr = sf.read(src, dtype="float32", always_2d=True)
        if sr == OUT_SR:
            shutil.copy2(src, dst)
        else:  # resample each channel, 16-bit wav like the source
            y = AF.resample(torch.from_numpy(x.T.copy()), sr, OUT_SR, **RESAMPLE).T.numpy().clip(-1.0, 1.0)
            sf.write(dst, y, OUT_SR, subtype="PCM_16")
        info = sf.info(dst)
        row = {
            "audio_filepath": str(dst) if ABSOLUTE_PATHS else f"{wav_dir.name}/{dst.name}",
            "duration": round(info.duration, 3),
            "sample_rate": info.samplerate,
            "speaker": f"{num:03d}_{r['speaker']}",  # speaker_0 is only unique inside one podcast
            "podcast": title,
            "start": r["start"],  # where the chunk is in the podcast audio (s)
            "end": r["end"],
            "original": str(src.relative_to(root)),
            **{k: r[k] for k in STATS if k in r},
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("saved", len(items), "chunks to", wav_dir, "and", dataset_dir / "metadata.jsonl")

100%|██████████| 14983/14983 [01:27<00:00, 171.28file/s]

saved 14983 chunks to /Users/macbook/Documents/IT/projects/tts/dataset/chunks and /Users/macbook/Documents/IT/projects/tts/dataset/metadata.jsonl


In [3]:
# check: metadata and folder match, and a summary of the dataset
meta = [json.loads(l) for l in (dataset_dir / "metadata.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
wavs = {p.name for p in wav_dir.glob("*.wav")}
listed = {Path(m["audio_filepath"]).name for m in meta}
print("rows:", len(meta), "| wavs:", len(wavs), "| wav without row:", len(wavs - listed), "| row without wav:", len(listed - wavs))
total = sum(m["duration"] for m in meta)
print(f"total {int(total // 3600)}h {int(total % 3600 // 60):02d}m, {len({m['speaker'] for m in meta})} speakers, "
      f"sample rates {dict(Counter(m['sample_rate'] for m in meta))}")
for lo, hi in zip([0, 1, 2, 5, 10, 20], [1, 2, 5, 10, 20, 30.5]):
    part = [m["duration"] for m in meta if lo <= m["duration"] < hi]
    print(f"  {lo:>2}-{hi:<4} s: {len(part):5d} chunks, {sum(part) / 3600:5.2f} h")

rows: 14983 | wavs: 14983 | wav without row: 0 | row without wav: 0
total 36h 46m, 114 speakers, sample rates {48000: 14983}
   0-1    s:  1147 chunks,  0.25 h
   1-2    s:  2153 chunks,  0.87 h
   2-5    s:  3623 chunks,  3.30 h
   5-10   s:  2735 chunks,  5.41 h
  10-20   s:  3356 chunks, 13.55 h
  20-30.5 s:  1969 chunks, 13.39 h
